In [9]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np


In [10]:
sold = pd.read_csv('sold.csv')
list = pd.read_csv('list.csv')

C:\Users\ericy\AppData\Local\Temp\ipykernel_24524\856572006.py:1: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: WaterfrontYN, 3: OriginatingSystemName, 4: OriginatingSystemSubName, 5: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv('sold.csv')
C:\Users\ericy\AppData\Local\Temp\ipykernel_24524\856572006.py:2: DtypeWarning: Columns (0: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  list = pd.read_csv('list.csv')


In [11]:
key_columns = ['ClosePrice', 'ListPrice', 'OriginalListPrice', 'LivingArea', 'LotSizeAcres', 'BedroomsTotal', 'BathroomsTotalInteger', 'DaysOnMarket', 'YearBuilt']

In [12]:
#sold.head()
#sold.columns
print(sold['PropertyType'].unique())
sold_missing_count = sold.isnull().sum()
sold_missing_pct = sold.isnull().mean()
sold_missing_table = pd.DataFrame({
    "Missing Count": sold_missing_count,
    "Missing %": sold_missing_pct,
}).iloc[1:,:]
#sold_missing_table.head()
sold_missing_table.to_csv('sold_missing_table.csv', index=False)

list_missing_count = list.isnull().sum()
list_missing_pct = list.isnull().mean()
list_missing_table = pd.DataFrame({
    "Missing Count": list_missing_count,
    "Missing %": list_missing_pct,
}).iloc[1:,:]
#list_missing_table.head()
list_missing_table.to_csv('list_missing_table.csv', index=False)

<StringArray>
[        'Residential',     'CommercialLease',                'Land',
    'ResidentialLease',  'ManufacturedInPark',   'ResidentialIncome',
      'CommercialSale', 'BusinessOpportunity']
Length: 8, dtype: str


In [13]:
cols_to_drop = [ #drop all columns that are not key category and over 90% missing
    col for col in sold.columns
    if sold_missing_pct[col] >= 0.90 and col not in key_columns
]
print('columns from sold with more than 90% missing:')
print(cols_to_drop)
soldfiltered = sold.drop(columns=cols_to_drop)[sold.PropertyType == 'Residential']
soldfiltered.to_csv('soldfiltered.csv', index=False)

cols_to_drop = [ #drop all columns that are not key category and over 90% missing
    col for col in list.columns
    if list_missing_pct[col] >= 0.90 and col not in key_columns
]
print('columns from list with more than 90% missing:')
print(cols_to_drop)
listfiltered = list.drop(columns=cols_to_drop)[list.PropertyType == 'Residential']
listfiltered.to_csv('listfiltered.csv', index=False)

columns from sold with more than 90% missing:
['WaterfrontYN', 'BasementYN', 'FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'BuilderName', 'TaxYear', 'ElementarySchoolDistrict', 'CoBuyerAgentFirstName', 'BelowGradeFinishedArea', 'BusinessType', 'CoveredSpaces', 'LotSizeDimensions', 'MiddleOrJuniorSchoolDistrict']
columns from list with more than 90% missing:
['FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'ElementarySchool', 'BuilderName', 'TaxYear', 'ElementarySchoolDistrict', 'CoBuyerAgentFirstName', 'BelowGradeFinishedArea', 'BusinessType', 'CoveredSpaces', 'MiddleOrJuniorSchool', 'LotSizeDimensions', 'MiddleOrJuniorSchoolDistrict']


In [14]:
soldfiltered_key_columns = soldfiltered[key_columns]
soldfiltered_summary = pd.DataFrame({
    "min": soldfiltered_key_columns.min(),
    "max": soldfiltered_key_columns.max(),
    "mean": soldfiltered_key_columns.mean(),
    "median": soldfiltered_key_columns.median(),
    "std%": soldfiltered_key_columns.std()
}).T
soldfiltered_summary.head()
soldfiltered_summary.to_csv('soldfiltered_summary.csv', index=False)

listfiltered_key_columns = listfiltered[key_columns]
listfiltered_summary = pd.DataFrame({
    "min": listfiltered_key_columns.min(),
    "max": listfiltered_key_columns.max(),
    "mean": listfiltered_key_columns.mean(),
    "median": listfiltered_key_columns.median(),
    "std%": listfiltered_key_columns.std()
}).T
listfiltered_summary.head()
listfiltered_summary.to_csv('listfiltered_summary.csv', index=False)

In [15]:
#historgrams
#key_columns = ['ClosePrice', 'ListPrice', 'OriginalListPrice', 'LivingArea', 'LotSizeAcres', 'BedroomsTotal', 'BathroomsTotalInteger', 'DaysOnMarket', 'YearBuilt']
for col in key_columns:
    x = soldfiltered[col].dropna()
    # Skip columns with non-positive values
    x = x[x > 0]
    if len(x) == 0:
        continue
    # Create logarithmically spaced bins
    bins = np.logspace(
        np.log10(x.min()),
        np.log10(x.max()),
        50
    )
    plt.hist(x, bins=bins)
    plt.xscale("log")     # logarithmic x-axis
    plt.yscale("log")     # logarithmic y-axis

    plt.title(col)
    plt.xlabel(col)
    plt.ylabel("Count")

    plt.tight_layout()
    plt.savefig(f"soldfiltered {col}.png", dpi=300)
    plt.close()
for col in key_columns:
    x = listfiltered[col].dropna()
    # Skip columns with non-positive values
    x = x[x > 0]
    if len(x) == 0:
        continue
    # Create logarithmically spaced bins
    bins = np.logspace(
        np.log10(x.min()),
        np.log10(x.max()),
        50
    )
    plt.hist(x, bins=bins)
    plt.xscale("log")     # logarithmic x-axis
    plt.yscale("log")     # logarithmic y-axis

    plt.title(col)
    plt.xlabel(col)
    plt.ylabel("Count")

    plt.tight_layout()
    plt.savefig(f"listfiltered {col}.png", dpi=300)
    plt.close()



In [16]:
#